In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
Path.cwd()

PosixPath('/home/avash/Documents/research/transformer-pt-analysis-main/src/Analysis')

### Training for several random seed at fixed p and lambda

In [3]:
import os
import torch
import random
from mintrans_clean import *


cfg = TrainConfig



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

files = os.listdir('./random_seeds')



seeds = set()

for _ in range(10):
    seed = random.randint(0, 100)
    while seed in seeds:
        seed = random.randint(0, 100)

    seeds.add(seed)

seeds = list(seeds)

# Fixed lambda and p with random seeds are stored here
checkpoint_dir = './random_seeds'
if not os.path.exists(checkpoint_dir): 
    os.makedirs(checkpoint_dir)


# Train and test datasets
train_ds = FibonacciTrainDataset(
    mod=cfg.p, seq_len=cfg.seq_len, train_frac=cfg.train_frac,
    seed=cfg.data_seed,
)
val_ds = FibonacciValDataset(train_ds, num_samples=500)


train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size,
    num_workers=4, pin_memory=True, persistent_workers=True,
)



# Full batch training
cfg.batch_size = len(train_ds)   
cfg.epochs = 10000

for seed in seeds:
    # the only thing different is we are doing this with random seeds.
    cfg.torch_seed = seed

    torch.manual_seed(cfg.torch_seed)


    # ---------------------------------------------------------------------------
    # Model configuration 
    # ---------------------------------------------------------------------------


    model = MinimalTransformer(
        vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
        num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
        hidden_mlp=cfg.hidden_mlp,
    ).to(device)

    # general_paramater_seed_<random_seed>.pth
    cfg.checkpoint_path = os.path.join(checkpoint_dir, f'general_paramater_seed_{seed}.pth')

    try:
        history = train_model(model, train_loader, val_loader, cfg)
    except KeyboardInterrupt:
        history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc", "spectral")}

    val_loss, val_acc     = evaluate(model, val_loader)
    train_loss, train_acc = evaluate(model, train_loader)

    print(f"\nFinal — train acc: {train_acc:.3f}  val acc: {val_acc:.3f}")

    if cfg.checkpoint_path:
        save_checkpoint(model, history, cfg.checkpoint_path)

Epoch   100/10000  train loss 1.0906 acc 0.816  val loss 9.1602 acc 0.014
Epoch   200/10000  train loss 0.0340 acc 1.000  val loss 14.8836 acc 0.020
Epoch   300/10000  train loss 0.0019 acc 1.000  val loss 18.8853 acc 0.022
Epoch   400/10000  train loss 0.0022 acc 1.000  val loss 17.3933 acc 0.026
Epoch   500/10000  train loss 0.0071 acc 1.000  val loss 14.5810 acc 0.030
Epoch   600/10000  train loss 0.0007 acc 1.000  val loss 18.2564 acc 0.034
Epoch   700/10000  train loss 0.0100 acc 1.000  val loss 13.0494 acc 0.032
Epoch   800/10000  train loss 0.0004 acc 1.000  val loss 18.3524 acc 0.026
Epoch   900/10000  train loss 0.0107 acc 1.000  val loss 12.4035 acc 0.032
Epoch  1000/10000  train loss 0.0005 acc 1.000  val loss 17.1472 acc 0.032
Epoch  1100/10000  train loss 0.0094 acc 1.000  val loss 12.2332 acc 0.036
Epoch  1200/10000  train loss 0.0003 acc 1.000  val loss 17.3139 acc 0.046
Epoch  1300/10000  train loss 0.0000 acc 1.000  val loss 21.9315 acc 0.052
Epoch  1400/10000  train l

### Finite-size scaling of the transition width for a range of primes (train)

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
Path.cwd()


import os
import torch
import random
from mintrans_clean import *


cfg = TrainConfig



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

files = os.listdir('./range_of_primes')


checkpoint_dir = './range_of_primes'
if not os.path.exists(checkpoint_dir): 
    os.makedirs(checkpoint_dir)

# We do not want to make this depend upon the above cell,
# if that executes then it might corrupt the existing checkpoint.

train_ds = FibonacciTrainDataset(
    mod=cfg.p, seq_len=cfg.seq_len, train_frac=cfg.train_frac,
    seed=cfg.data_seed,
)
val_ds = FibonacciValDataset(train_ds, num_samples=500)


train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

# Full batch training
cfg.batch_size = len(train_ds)   
cfg.epochs = 7500


cfg.p = 149

# ---------------------------------------------------------------------------
# Model configuration 
# ---------------------------------------------------------------------------


model = MinimalTransformer(
    vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
    num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
    hidden_mlp=cfg.hidden_mlp,
).to(device)

# general_paramater_p_<p>.pth
cfg.checkpoint_path = os.path.join(checkpoint_dir, f'general_paramater_p_{cfg.p}.pth')

try:
    history = train_model(model, train_loader, val_loader, cfg)
except KeyboardInterrupt:
    history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc", "spectral")}

val_loss, val_acc     = evaluate(model, val_loader)
train_loss, train_acc = evaluate(model, train_loader)

print(f"\nFinal — train acc: {train_acc:.3f}  val acc: {val_acc:.3f}")

if cfg.checkpoint_path:
    save_checkpoint(model, history, cfg.checkpoint_path)


Epoch   100/7500  train loss 3.2343 acc 0.308  val loss 7.0519 acc 0.018
Epoch   200/7500  train loss 0.0583 acc 1.000  val loss 15.5091 acc 0.110
Epoch   300/7500  train loss 0.0138 acc 1.000  val loss 16.3788 acc 0.132
Epoch   400/7500  train loss 0.0045 acc 1.000  val loss 17.1241 acc 0.156
Epoch   500/7500  train loss 0.0015 acc 1.000  val loss 18.0133 acc 0.172
Epoch   600/7500  train loss 0.0005 acc 1.000  val loss 18.9196 acc 0.184
Epoch   700/7500  train loss 0.0002 acc 1.000  val loss 19.7936 acc 0.190
Epoch   800/7500  train loss 0.0001 acc 1.000  val loss 20.6308 acc 0.194
Epoch   900/7500  train loss 0.0000 acc 1.000  val loss 21.4046 acc 0.210
Epoch  1000/7500  train loss 0.0000 acc 1.000  val loss 22.1198 acc 0.220
Epoch  1100/7500  train loss 0.0000 acc 1.000  val loss 22.6822 acc 0.224
Epoch  1200/7500  train loss 0.0000 acc 1.000  val loss 23.0640 acc 0.236
Epoch  1300/7500  train loss 0.0000 acc 1.000  val loss 23.1550 acc 0.234
Epoch  1400/7500  train loss 0.0000 acc